Quick and dirty notebook for simulation of coherent scattering data of magnetic samples in fraunhofer far-field regime

# Import

In [ ]:
# Import general libraries
import sys, os
from os.path import join, split
from importlib import reload
from copy import deepcopy
from tqdm.auto import tqdm

import numpy as np

# scipy
import scipy

# plotting
import matplotlib.pyplot as plt

# Interactive plotting
import ipywidgets
%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

In [ ]:
# Imports from our own codebase
from scattering_calculator.experimental_conditions import detector, light_beam
from scattering_calculator.sample_generator import pattern_generator, structures
from scattering_calculator.beam_propagator import Jones_propagator
from scattering_calculator.utils import masking,physics,image_transformator
from scattering_calculator.interactive.interactive_widgets import cimshow

### EXPERIMENTAL GEOMETRY

In [ ]:
# ===================
# X-ray Source
# ===================
x_ray_energy = 789.9  # eV
x_ray_photon_flux = 1e10  # Photons per pulse
beam_params = light_beam.beam_parameters(
    x_ray_energy, x_ray_photon_flux
)

# ==================
# Geometry
# ==================
detector_pixel_size = 10e-6  # in m
detector_pixel_shape = (1024,1024)
detector_distance = 0.15  # in m

# Optional: Define a beamstop
beamstop_radius = 0.5e-3  # in m
beamstop_distance = 0.001  # in m
beamstop_center = np.array(detector_pixel_shape) // 2  # in px

# ==================
# Setup
# ==================

# Basic camera parameters
exp_detector = detector.detector_layout(
    pixel_size=detector_pixel_size,
    shape=detector_pixel_shape,
    distance_sample_detector=detector_distance,
)

# Add beamstop to detector layout
beamstop = detector.beamstop(exp_detector, beamstop_distance)
beamstop.create_circle_beamstop(beamstop_center, beamstop_radius,use_real_space_coordinates=True)
bs_mask = beamstop.return_beamstop()
exp_detector.assign_beamstop(bs_mask)

### SAMPLE STACK STRUCTURE AND OPTICAL PROPERTIES

In [ ]:

reload(structures)


recipe = "Au(1000)/Co(100)/Pt(2)"
#"Au(1000)/SiN(200)/Ta(5)/[Pt(1)/Co(1)]x15/Pt(3)"

# define stack
stack = structures.parse_recipe(
    recipe,
    sample_name="sample_A",
    comments=["test multilayer"],
)

#pulls material refractive indexes
material_params = structures.material_params(materials=set([element.material for element in stack.layers]), 
                             x_ray_energy=x_ray_energy)


non_magnetic_structure = structures.Structure(
    name="Test Structure",
    material_params=material_params
)

for layer in stack.layers:
    non_magnetic_structure.add_layer(layer.material, thickness=layer.thickness)

dielectric_tensor=np.array(non_magnetic_structure.dielectric_tensors)


# SAMPLE 

In [ ]:
# Basic parameters for the simulation
sample_shape = (len(stack.layers),2**10, 2**10)  # in pixels
real_space_pixel_size = 1e-9  # in m


### - magnetic domains

In [ ]:

reload(pattern_generator)

%time
# Skyrmion and Screening diameter
skyrmion_radius = 40e-9  # m
screening_radius = (
    1.5 * skyrmion_radius
)  # m, either only even or odd, otherwise script will fail
skyrmion_smoothing = 3

# Number of skyrmions
# If this number is too high, the script may take forever ...
# (brute force algorithm)
number_of_skyrmions = 150000
max_nr_iteration = 1000  # 10 * sample["no_skyr"]

# Create Pattern
skyrmion_pattern, coordinates = pattern_generator.create_skyrmion_pattern(
    sample_shape[1:],
    skyrmion_radius/real_space_pixel_size,
    screening_radius/real_space_pixel_size,
    number_of_skyrmions,
    max_nr_iteration,
    sigma=skyrmion_smoothing,
    real_space_pixel_size = real_space_pixel_size,
    plot=True,
)

magnetization=pattern_generator.map_magnetization_to_3d(0*skyrmion_pattern,np.sqrt(1-np.abs(skyrmion_pattern)**2),skyrmion_pattern,nr_repeats=sample_shape[0])



In [ ]:
fig,ax=plt.subplots(1,3,figsize=(10,5))
ax[0].imshow( np.sum(magnetization[...,2], axis=0))

# - holography mask

In [ ]:
np.sum(non_magnetic_structure.layer_thicknesses)

In [ ]:
reload(structures)
reload(masking)


front_aperture = structures.Apertures3D(sample_shape, real_space_pixel_size,
    layer_thicknesses=non_magnetic_structure.layer_thicknesses)

front_aperture_radius = 100e-9  # in m
front_aperture.create_circle_aperture(
    center=(sample_shape[1] // 2, sample_shape[2] // 2),
    depth=1000e-9,
    radius=front_aperture_radius,
    use_real_space_coordinates=True,
    sigma=10e-9
)


front_aperture_radius = 10e-9  # in m
front_aperture.create_circle_aperture(
    center=(sample_shape[1] // 2+200, sample_shape[2] // 2+200),
    depth=np.sum(non_magnetic_structure.layer_thicknesses),
    radius=front_aperture_radius,
    use_real_space_coordinates=True,
    sigma=10e-9
)


front_aperture.visualize_aperture()

mask=front_aperture.aperture_design

### ILLUMINATION FUNCTION

In [ ]:
# Params for gaussian beam
illumination_function = "gaussian"
illumination_center = np.array(sample_shape[1:]) // 2  # in px
illumination_focus_distance = 0#in m
illumination_fwhm = 50e-6  # in m

illumination = light_beam.illumination(beam_params, sample_shape[1:], real_space_pixel_size)

# Comment: Check gauss_beam function for different focus distances
illumination.gauss_beam(
    illumination_center,
    illumination_focus_distance,
    illumination_fwhm,
)
illumination_wavefield = illumination.return_illumination()
extent_illumination_real = illumination.get_illumination_extent_real_space()
illumination.visualize_illumination()

### Do light propagation

In [ ]:
dielectric_tensor_vacuum=np.array([[1.+0.j, 0.+0.j],[0.+0.j, 1+0.j]])

final_dielectric_tensor=np.einsum("ij,zyx->zyxij", dielectric_tensor_vacuum, 1-mask)+np.einsum("zij,zyx->zyxij", dielectric_tensor[:,0,:,:], mask)+np.einsum("zij,zyx->zyxij", dielectric_tensor[:,1,:,:], magnetization[:,:,:,2])+np.einsum("zij,zyx->zyxij", dielectric_tensor[:,2,:,:], magnetization[:,:,:,0])-np.einsum("zij,zyx->zyxij", dielectric_tensor[:,2,:,:], magnetization[:,:,:,1])


fig,ax=plt.subplots()
ax.imshow( np.sum(np.abs(final_dielectric_tensor), axis=(0,3,4)))

In [ ]:
reload(Jones_propagator)


polarization_vector = np.array([1, -1j]) / np.sqrt(2)  # Circular polarization
illwave=np.einsum("yx,s->yxs", illumination_wavefield, polarization_vector)
print(illwave.shape)

E_out_p = Jones_propagator.propagate_jones_multislice(
    illumination=illwave,
    eps_stack=np.array(final_dielectric_tensor),
    wavelength=physics.photon_energy_wavelength(x_ray_energy),
    thicknesses=non_magnetic_structure.layer_thicknesses,
    pixel_size=real_space_pixel_size,
    propagate=False
)

if True:

    polarization_vector = np.array([1, +1j]) / np.sqrt(2)  # Circular polarization
    illwave=np.einsum("yx,s->yxs", illumination_wavefield, polarization_vector)

    E_out_n = Jones_propagator.propagate_jones_multislice(
        illumination=illwave,
        eps_stack=np.array(final_dielectric_tensor),
        wavelength=physics.photon_energy_wavelength(x_ray_energy),
        thicknesses=non_magnetic_structure.layer_thicknesses,
        pixel_size=real_space_pixel_size,
        propagate=False
    )


In [ ]:

def E_I(E,pol):
    I= (np.sum(np.abs(E_j(E, pol))**2, axis=(2)))
    return I

fig,ax=plt.subplots(1,5,figsize=(10,3))
ax[0].imshow( np.sum(final_dielectric_tensor, axis=(0,3,4)).real)
ax[1].imshow( np.sqrt(2)*np.abs(illumination_wavefield)**2)
ax[2].imshow( np.abs(E_out_p[:,:,0]) )
ax[3].imshow( E_I(E_out_n, "-") )
ax[4].imshow(  E_I(E_out_p, "+")- E_I(E_out_n, "-"))

In [ ]:
reload(image_transformator)
holo_p=image_transformator.Fraunhofer_propagation_jones(E_out_p)
I_p=E_I(holo_p, '+')

def E_I(E,pol):
    I= np.sqrt(np.sum(np.abs(E_j(E, pol))**2, axis=(2)))
    return I

fig,ax=plt.subplots()
ax.imshow(np.log10(I_p))


FTH=image_transformator.reconstruct(I_p)
fig,ax=plt.subplots()
ax.imshow(np.real(FTH))

In [ ]:
fig,ax=plt.subplots()
ax.imshow(np.abs(E_out_p[:,:,0]))
fig,ax=plt.subplots()
ax.imshow(np.abs(holo_p[:,:,0]))


In [ ]:
def E_j(E, pol="+"):
    if pol=="+":
        polarization_vector = np.array([1, -1j]) / np.sqrt(2)
    elif pol=="-":
        polarization_vector = np.array([1, +1j]) / np.sqrt(2)

    return np.einsum("yxs,s->yxs", E, polarization_vector)

def E_I(E,pol):
    I= (np.sum(np.abs(E_j(E, pol))**2, axis=(2)))
    return I


print(E_I(E_out_p, "+")[528,538])
print(E_j(E_out_p, "+")[528,538])
print(E_out_p[528,538])



In [ ]:
def E_j(E, pol="+"):
    if pol=="+":
        polarization_vector = np.array([1, -1j]) / np.sqrt(2)
    elif pol=="-":
        polarization_vector = np.array([1, +1j]) / np.sqrt(2)

    return np.einsum("yxs,s->yx", E, polarization_vector)

def E_I(E,pol):
    I= np.sqrt(np.abs(E_j(E, pol))**2)
    return I


print(E_I(E_out_p, "+")[528,538])
print(E_j(E_out_p, "+")[528,538])
print(E_out_p[528,538])

In [ ]:
fig,ax=plt.subplots(1,5,figsize=(10,3))
ax[0].imshow( final_dielectric_tensor[2,...,1,0].real)
ax[1].imshow( np.sqrt(2)*np.abs(illumination_wavefield)**2)
ax[2].imshow( E_I(E_out_p, "+") )
ax[3].imshow( np.abs(E_j(E_out_n, "-") ))
ax[4].imshow(  E_I(E_out_p, "+")- E_I(E_out_n, "-"))